# Managed Agents: The Agents API

Create a reusable agent, run a task in an OpenAI-hosted environment, continue its session, and download its output files.

[Open in Google Colab](https://colab.research.google.com/github/BrightPool/udemy-prompt-engineering-course/blob/main/openai_features_and_functionality/managed_agents_api.ipynb) · [Official quickstart](https://developers.openai.com/api/docs/guides/agents-api/quickstart)

**Harness as a service:** OpenAI manages the agent loop, durable sessions, context compaction, and recovery. We provide the task and choose the tools and environment.

| Concept | What it holds |
|---|---|
| Saved agent | Reusable model and instructions |
| Session | This task's conversation and saved work |
| Turn | One cycle of work within that session |
| Environment | The sandbox where commands and file edits run |
| Artifacts | Files the agent publishes from `/workspace/outputs` |

The current Python interface is `client.beta.agents`, distinct from the local **Agents SDK** (`Agent` / `Runner`). You need Agents API access and an application key with `api.agents.read`, `api.agents.write`, and `api.responses.write`. The SDK supplies the beta header. Model and hosted-environment usage incur API charges. This lesson uses synthetic sales data and no external tools or UI automation.

In [ ]:
%pip install openai --upgrade --quiet

In [ ]:
MODEL = "gpt-6-astra"  # Update this one line when a newer model ships.

In [ ]:
import os
from getpass import getpass
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI(timeout=180, max_retries=1)
if not hasattr(client.beta, "agents"):
    raise RuntimeError("Update the openai package, then restart the runtime and rerun this cell.")
print("Model:", MODEL)

## 1. Define a reusable agent

The saved agent is a configuration, not a conversation: a model plus instructions. We give it a narrow role and ask it to report actual execution results.

In [ ]:
agent = client.beta.agents.create(
    model=MODEL,
    instructions=(
        "You are a careful data analyst. Use Python's standard library for calculations. "
        "Run the code and check the actual output. Save requested files under "
        "/workspace/outputs. Do not use network access or delegate work. "
        "Use plain text without em dashes."
    ),
)
print("Saved agent:", agent.id)

## 2. A helper to follow a turn

A session streams **events** while the agent works. This helper prints the agent's text as it arrives and returns when the turn completes. It also picks up the **session ID** from the first event, which we need for follow-ups and downloads.

In [ ]:
def follow_turn(events, session_id=None):
    """Print a turn's text as it streams. Returns (session_id, turn_id)."""
    for event in events:
        e = event.model_dump()
        kind = e["type"]

        if kind == "agent.session.created":
            session_id = e["session"]["id"]
            print("Session:", session_id)
        elif kind == "agent.session.turn.output_text.delta":
            print(e["delta"], end="", flush=True)
        elif kind in {"error", "agent.session.failed", "agent.session.environment.failed",
                      "agent.session.turn.failed", "agent.session.turn.cancelled"}:
            raise RuntimeError(f"The agent stopped: {e}")
        elif kind.endswith("requires_action"):
            raise RuntimeError("The agent is waiting for an action this demo does not handle.")
        # Sub-agents send their own completion events; only the main turn ends the loop.
        elif kind == "agent.session.turn.completed" and not (e.get("turn") or {}).get("subagent_id"):
            print("\nTurn completed:", e["turn_id"])
            return session_id, e["turn_id"]

    raise RuntimeError("The stream ended early. Retrieve the session before retrying.")

## 3. Start a session and let the managed harness work

The task needs several steps: write code, run it, and save two files. OpenAI's harness runs that loop inside an **OpenAI-hosted environment**, so we only send the task and watch the events.

In [ ]:
TASK = (
    "Create and run a Python script that totals this synthetic revenue: "
    "North: 120, 140, 160. South: 100, 120, 140. "
    "Write /workspace/outputs/regional_totals.csv with exactly the columns "
    "region,total_revenue and one row per region. "
    "Write /workspace/outputs/weekly_report.txt with regional totals and the grand total. "
    "Show the script's actual output and check the arithmetic. Keep the task brief."
)

with client.beta.agents.sessions.create(
    agent_id=agent.id,
    environment={"type": "openai_hosted"},
    input=TASK,
    stream=True,
) as events:
    session_id, turn_id = follow_turn(events)

## 4. Continue the same session

A follow-up is a new **turn** in the same session, so the agent still has the conversation and its files. We open the event stream **before** sending the message so we do not miss early events.

In [ ]:
FOLLOW_UP = (
    "Continue from the existing files. Add one line naming the region with the "
    "highest revenue to weekly_report.txt. Rerun the calculation and save both "
    "files under /workspace/outputs. Report the actual results briefly."
)

with client.beta.agents.sessions.events.stream(session_id) as events:
    client.beta.agents.sessions.events.create(
        session_id,
        events=[{
            "type": "agent.session.input.message",
            "input": [{"role": "user", "content": [{"type": "input_text", "text": FOLLOW_UP}]}],
        }],
    )
    session_id, turn_id = follow_turn(events, session_id)

## 5. Download the output files

When a turn completes, files under `/workspace/outputs` are published as **artifacts**. We take the newest version of each file and save it locally.

In [ ]:
from pathlib import Path
from IPython.display import display, FileLink

workdir = Path("managed_agents_demo")
workdir.mkdir(exist_ok=True)

artifacts = list(client.beta.agents.sessions.artifacts.list(session_id, order="desc"))
for filename in ("weekly_report.txt", "regional_totals.csv"):
    match = next(a for a in artifacts if a.path == f"/workspace/outputs/{filename}")
    with client.beta.agents.sessions.artifacts.with_streaming_response.content(
        match.id, session_id=session_id,
    ) as content:
        content.stream_to_file(workdir / filename)
    display(FileLink(str(workdir / filename)))

## 6. Check the result

A completed turn means the agent finished, not that it was right. We check its numbers against the totals we can work out ourselves: North 420, South 360.

In [ ]:
import csv

print((workdir / "weekly_report.txt").read_text())

with (workdir / "regional_totals.csv").open() as f:
    actual = {row["region"]: float(row["total_revenue"]) for row in csv.DictReader(f)}
assert actual == {"North": 420, "South": 360}, f"Calculation mismatch: {actual}"
print("Verified totals:", actual, "Grand total:", sum(actual.values()))

## 7. Clean up

Delete this demo's session and saved agent. The downloaded files stay in `managed_agents_demo/`. Skip this cell if you want to keep the session and try another follow-up first.

In [ ]:
client.beta.agents.sessions.delete(session_id)
client.beta.agents.delete(agent.id)
print("Deleted this demo's session and saved agent; local files remain.")

## Try it yourself

Keep a session, then ask for a percentage breakdown in a second turn. Inspect the new artifact version and check that percentages add up to 100%. Explore passing input files through `environment.files` in the linked guide.

## Summary

- A saved agent holds reusable configuration; a session holds one continuing conversation and its work.
- The Agents API supplies a managed Codex harness for orchestration, compaction, and recovery.
- Choose an OpenAI-hosted sandbox to run code and edit files without implementing the executor yourself.
- Stream progress, reuse the session ID for follow-ups, and verify downloaded artifacts.
- This hosted API is distinct from using the Agents SDK in your own Python process.

**Troubleshooting:** a 403 may mean your project lacks Agents API access or the required permissions. Update the SDK if `beta.agents` is missing. On a disconnected stream, retrieve `client.beta.agents.sessions.retrieve(session_id)` and list saved `sessions.items` before deciding whether to continue. If a cell fails partway, run the cleanup cell once you have saved what you need.

Sources: [Quickstart](https://developers.openai.com/api/docs/guides/agents-api/quickstart), [Agent configuration](https://developers.openai.com/api/docs/guides/agents-api/configuration), [Sessions](https://developers.openai.com/api/docs/guides/agents-api/sessions), [Events](https://developers.openai.com/api/docs/guides/agents-api/sessions/events), [Files and artifacts](https://developers.openai.com/api/docs/guides/agents-api/environments/files).